# 14d · 把两个能量变成一张位点地图

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F：一条线，从一个分数到两个能量]
                 ├─ 14a 把稳定性变成能量 ✅
                 ├─ 14b 把测序读数变成两种测量 ✅
                 ├─ 14c 把一个分数拆成两个能量 ✅
                 ├─ 14d 把两个能量变成一张位点地图  ← 你在这里
                 ├─ 14e 把地图用到一个真实的药上 ✅
                 └─ 14f 把整条线讲出来 ✅

附录（不在这条线上）
                 └─ 14x 实验室该测多少个 ✅
```

## 背景：从"这个突变怎么样"到"这个位置是干什么的"

前面几章都是在回答**突变**的问题。但药物研发更常问的是**位置**的问题：

> 这个靶点上，哪些地方是结构支架（动不得）、哪些地方是干活的（可以当药物口袋）、哪些地方随便改（可以挂标签、做改造）？

14c 的两个能量可以直接回答这个。把一个位置上 19 种替换的能量取平均，得到这个**位置**的两个代价：

| 位置类型 | 折叠代价 | 功能代价 | 这个位置是什么 | 对项目意味着什么 |
|---|---|---|---|---|
| **结构关键** | 高 | 低 | 蛋白的骨架，埋在里面 | 动不得；突变坏了可以试试伴侣分子扶住它 |
| **功能关键** | 低 | 高 | 折得好好的，但这里一动就不工作 | **活性位点、配体口袋、结合界面——候选药物口袋** |
| **两者都关键** | 高 | 高 | 又埋又重要 | 典型的催化中心 |
| **可以改** | 低 | 低 | 表面、不重要 | 挂荧光标签、改溶解度、做改造的地方 |

"**功能关键**"这一类是重点：它是"蛋白好好的，但不工作"的位置，也是小分子有可能结合的地方。

## 这一章做什么

| 步骤 | 内容 |
|---|---|
| 1 | 给一批蛋白的每个位置算两个代价，贴标签 |
| 2 | **对不对**：和 UniProt 注释的活性位点、配体位点比 |
| 3 | **有没有用**：三个**已经成药**的变构口袋，模型事先点不点得亮 |
| 4 | **更多检验**：和实测的四类对照、102 个蛋白的蛋白量、BioLiP 结合残基、21 个上市药的口袋 |
| 5 | 地图进数据湖，能用 SQL 查 |
| 6 | 接进 API 和 agent |
| 7 | 记账 |

---

## 准备

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from qafi_platform import athena, lake, tracking
from qafi_platform.lakehouse import sql
from qafi_core import energy, sites, stability, validation
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
WORK, MODELS, MAP = ROOT / "data/stability", ROOT / "data/models", ROOT / "data/sites"
MAP.mkdir(parents=True, exist_ok=True)
print("处理器:", stability.device())

---

## 第 1 步 · 给每个位置贴标签

### 在干什么
用 14c 的模型，给一批蛋白的**每个位置的每一种替换**算两个能量，再按位置平均，贴上标签。

### 哪些蛋白
- **108 个锁死的考试蛋白**：模型没见过，标签才算数
- 加上三个**有成药变构口袋**的蛋白：KRAS、ABL1、SHP2（第 3 步要用）

### 怎么定"高"和"低"
阈值**按蛋白各算各的**（每个蛋白最差的 25%）。因为蛋白之间的整体容忍度差很多，而这里要问的是"在这个蛋白**内部**，哪些位置突出"。

### 执行
训练模型（和 14c 一样），算特征，给约 200 万个替换打分。约 10 分钟。

In [ ]:
POCKETS = {    # 已上市或在临床的药结合的口袋，残基取自沉积的结构
    "P01116": ("KRAS switch-II（sotorasib，6OIM）",
               [9, 10, 12, 16, 34, 58, 59, 60, 61, 62, 63, 68, 72, 95, 96, 99, 103]),
    "P00519": ("ABL1 肉豆蔻酸口袋（asciminib，5MO4）",   # 结构用 1b 编号 = UniProt 1a + 19
               [332, 337, 340, 341, 344, 429, 432, 433, 434, 435, 462, 464, 465, 468, 493, 502, 506, 510]),
    "Q06124": ("SHP2 通道（SHP099，5EHR）",
               [108, 110, 111, 113, 114, 218, 219, 249, 250, 253, 254, 257, 491, 492, 495]),
}
splits = stability.load_splits(WORK / "mega_splits.pkl")
mega = stability.add_features(
    stability.megascale(WORK / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv", splits),
    pd.read_parquet(WORK / "structure.parquet"), pd.read_parquet(WORK / "residue_index.parquet"),
    np.load(WORK / "logprobs.npy"))
mega = mega[mega.split.isin(["train", "val"])]
split_table = pd.read_csv(ROOT / "dbt/seeds/protein_split.csv")
sequences = pd.read_parquet(MODELS / "proteome_sequences.parquet").set_index("uniprot_id").sequence
locked = [sequences[u] for u in split_table.loc[split_table.split == "test", "uniprot_id"] if u in sequences.index]
mega = mega[~mega.domain.isin({d for d, s in mega.groupby("domain").seq.first().items()
                               if any(validation.related(s, t) for t in locked)})]
labels = pd.read_parquet(MODELS / "train_variants.parquet")
titles = sql(w, "SELECT dataset_id, title FROM qafi_dev.gold.dim_label_dataset").set_index("dataset_id").title.to_dict()
labels["reads"] = [energy.readout_kind(k, titles.get(d))
                   for k, d in zip(labels.data_kind, labels.dataset_id, strict=True)]
kinds = labels.groupby("uniprot_id").reads.apply(set)
paired = {p for p, k in kinds.items() if {energy.ABUNDANCE, energy.ACTIVITY} <= k}
rows, vectors = energy.stack(mega, np.load(WORK / "vectors.npy"), labels,
                             pd.read_parquet(MODELS / "train_embeddings.parquet"),
                             np.asarray(np.load(MODELS / "train_embeddings.npy", mmap_mode="r")), titles)
readout = rows[(rows.kind == energy.DDG) | rows.protein.isin(paired)]
models = [energy.fit(readout, vectors, energy.FEATURES, seed=s) for s in (0, 1, 2)]
print(f"模型训练完成（{len(paired)} 个成对蛋白的读数层）")

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "databricks" / "src"))
from lakehouse import language_model as lm
wanted = sorted(set(POCKETS) | set(split_table.loc[split_table.split == "test", "uniprot_id"]))
residues = pd.read_parquet(MODELS / "proteome_residues.parquet", filters=[("uniprot_id", "in", wanted)])
have = sorted(residues.uniprot_id.unique())
seqs = [sequences[u] for u in have]
model, tokenizer = lm.load()
read = lm.read_proteins(seqs, model, tokenizer, embeddings=True)
logprobs = np.concatenate([r[0] for r in read])
protein_vectors = np.concatenate([r[1] for r in read]).astype(np.float16)
index = pd.DataFrame({"domain": np.repeat(have, [len(s) for s in seqs]),
                      "position": np.concatenate([np.arange(1, len(s) + 1) for s in seqs])})
structure = residues[["uniprot_id", "position", "aa", "plddt", "rsa", "contacts_8", "contacts_12",
                      "hse_up", "hse_down"]].rename(columns={"uniprot_id": "domain"})
every = pd.DataFrame({
    "domain": np.repeat(index.domain.values, 20), "position": np.repeat(index.position.values, 20),
    "wt_aa": np.repeat(index.merge(structure, on=["domain", "position"], how="left").aa.values, 20),
    "mut_aa": np.tile(list(energy.AMINO_ACIDS), len(index))}).dropna(subset=["wt_aa"])
every = every[every.wt_aa != every.mut_aa].assign(ddg=0.0)
every = stability.add_features(every, structure, index, logprobs, strict=False)
fold, func = [], []
for start in range(0, len(every), 200_000):
    part = every.iloc[start:start + 200_000]
    f, g = zip(*(energy.energies(m, s, part, protein_vectors, energy.FEATURES) for m, s, _ in models))
    fold.append(np.mean(f, axis=0)); func.append(np.mean(g, axis=0))
every["ddg_fold"], every["ddg_func"] = np.concatenate(fold), np.concatenate(func)
every = every.rename(columns={"domain": "uniprot_id"})
positions = sites.classify(every)
positions.to_parquet(MAP / "positions.parquet")
print(f"{len(have)} 个蛋白、{len(positions):,} 个位置、{len(every):,} 个替换")
positions.position_is.value_counts().rename("位置数")

### 结果怎么看
118 个蛋白、103,744 个位置、197 万个替换。四类的数量：可以改最多（约 7 万），结构关键和功能关键各约 8 千，两者都关键约 1.8 万。

按定义每类的比例是固定的（各取最差 25%），所以数量本身不说明问题——**下面两步才是检验**。

---

## 第 2 步 · 对不对：和 UniProt 的注释比

### 在干什么
UniProt 为人工审核过的蛋白标注了**活性位点**、**配体结合位点**和其他功能位点。我们的"功能关键"位置，应该比别的位置更容易落在这些注释上。

### 这是什么性质的检验
这是**事后验证**，不是训练目标——模型从没见过任何位点注释，它只见过突变的测量值。如果"功能关键"确实富集在注释位点上，说明两个能量的拆分抓到了真东西。

### 执行

In [ ]:
uniprot = pd.read_csv("https://rest.uniprot.org/uniprotkb/stream?query=reviewed:true+AND+organism_id:9606"
                      "&fields=accession,ft_act_site,ft_binding,ft_site&format=tsv", sep="\t")
annotated = sites.known_sites(uniprot)
print(f"UniProt 注释了 {len(annotated):,} 个功能位置，覆盖 {annotated.uniprot_id.nunique():,} 个蛋白")
sites.enrichment(positions, annotated).round(4)

### 结果怎么看

| 位置类型 | 位置数 | 其中有注释 | 比例 | 比值比 | p |
|---|---|---|---|---|---|
| **两者都关键** | 17,831 | 186 | **1.04%** | **9.33** | ≈0 |
| **功能关键** | 8,153 | 53 | **0.65%** | **2.71** | ≈0 |
| 结构关键 | 8,153 | 5 | 0.06% | 0.21 | 1.0 |
| 可以改 | 69,607 | 39 | 0.06% | 0.08 | 1.0 |

方向完全正确：

- **"两者都关键"富集最强**（9.3 倍）。这符合生物学——催化中心通常既埋得深又不能动，两个能量都高。
- **"功能关键"富集 2.7 倍**，而**"结构关键"反而被稀释**（0.21 倍）。这正是拆分有意义的证据：如果两个能量只是同一个数的两个副本，它们对注释位点的富集应该一样，而不是一个富集一个稀释。
- "可以改"几乎不含注释位点（0.08 倍）——它确实是可以动的地方。

绝对比例都很低（1% 量级），因为 UniProt 只给一小部分位置做了注释（10 万个位置里只有 283 个有注释）。**比值比才是要看的数字。**

### 面试怎么说
> "The two energies were never shown a site annotation, yet positions the model calls functionally costly are enriched 2.7-fold for UniProt active and binding sites, those costly in both energies 9.3-fold, while structurally costly positions are depleted. A single score cannot produce that split."

---

## 第 3 步 · 有没有用：已经成药的变构口袋

### 在干什么
最硬的检验：拿三个**已经有药物结合**的变构口袋，看模型在不知情的情况下，会不会把这些残基排在"功能敏感"的前面。

### 三个口袋

| 蛋白 | 口袋 | 药 | 为什么是好例子 |
|---|---|---|---|
| **KRAS** | switch-II 口袋 | sotorasib（已上市） | 几十年"不可成药"的靶点，靠这个口袋破局 |
| **ABL1** | 肉豆蔻酸口袋 | asciminib（已上市） | 变构抑制剂，不碰 ATP 口袋，所以不怕常见耐药突变 |
| **SHP2** | 三结构域交界的通道 | SHP099（临床） | 变构位点的典型 |

残基编号取自沉积的晶体结构（配体 4 Å 以内的重原子接触）。**注意 ABL1**：结构用的是 1b 异构体编号，比 UniProt 的 1a 大 19，这里已经换算过——这类编号错位是最容易悄悄毁掉一个验证的坑。

### 怎么算过关
把一个蛋白自己的所有位置按"功能代价"排序，看口袋残基排在什么百分位（0 = 最敏感，1 = 最不敏感）。口袋应该明显靠前。

### 执行

In [ ]:
found = []
for uniprot_id, (what, pocket) in POCKETS.items():
    out = sites.pocket_rank(positions, uniprot_id, pocket)
    found.append({"口袋": what, "残基数": out["pocket residues"], "蛋白位置数": out["protein positions"],
                  "口袋的中位百分位": out["median percentile"],
                  "其他位置": out["median percentile elsewhere"], "p": out["p"]})
pd.DataFrame(found).round(4)

### 结果怎么看

| 口袋 | 残基 | 口袋的中位百分位 | 其他位置 | p |
|---|---|---|---|---|
| **KRAS switch-II**（sotorasib） | 17 | **0.233** | 0.532 | **0.0012** |
| **ABL1 肉豆蔻酸口袋**（asciminib） | 18 | **0.235** | 0.506 | **0.0017** |
| SHP2 通道（SHP099） | 15 | 0.506 | 0.498 | 0.41 |

**三个里中了两个。** KRAS 和 ABL1 这两个口袋的残基，被模型排在自己蛋白里功能最敏感的前 23%——而模型只见过突变测量，从没见过任何药物、任何口袋。

SHP2 没有信号。这值得说清楚：SHP2 的通道位于**三个结构域的交界处**，它的作用是维持蛋白的自抑制构象，而不是直接参与催化或结合配体。我们的"功能能"学的是"折好之后还能不能干活"，对这种**构象开关**类型的位点天然不敏感。这是这套方法的一个真实边界。

### 面试怎么说
> "As a prospective check I took three allosteric pockets that marketed or clinical drugs bind and asked where their residues rank inside their own protein. KRAS switch-II and the ABL1 myristoyl pocket land in the top quarter by functional cost, p around 0.001, with the model having seen no drug and no pocket. SHP2's tunnel does not — it is a conformational switch at a domain junction, which is outside what a folded-and-functional model represents."

---

## 第 4 步 · 更多检验

### 在干什么
第 2、3 步的证据有两个弱点：UniProt 的注释太少（10 万个位置里只有 283 个），成药口袋只有 3 个。这一步从四个方向补证据，用的全是模型没见过的蛋白：

| 检验 | 拿什么当真实值 | 检验的是 |
|---|---|---|
| 4.1 | 实验测的稳定性和功能（KRAS、CYP2C9、G6PD） | 四类整体对不对 |
| 4.2 | 102 个测试蛋白的蛋白量实验 | 稳定性这一轴 |
| 4.3 | BioLiP：晶体结构里实际接触配体的残基 | 功能这一轴 |
| 4.4 | 二十多个上市药物的结合口袋 | 有没有用 |

---

### 4.1 和实测的四类对照

**在干什么**：用实验测量给每个位置分出"真实的四类"，再和预测的四类比。

**真实值怎么来**：
- KRAS：14c 用过的 Weng 2024 数据。每个位置的实测折叠能，和 6 个结合能的平均
- CYP2C9、G6PD：蛋白量当稳定性；活性里蛋白量解释不了的部分当功能（和 14c 一样，`sites.beyond_abundance`）

两边用同一个规则分类（各取最差的 25%，`sites.position_classes`），只比有测量的位置。

**怎么判断**：一致率和随机分的一致率比；kappa 表示超出随机的部分（0 = 和随机一样，1 = 完全一致）；置换检验给 p。

**执行**

In [ ]:
from scipy import stats
positions = pd.read_parquet(MAP / "positions.parquet")      # 第 1 步的地图，和湖里的是同一份
test = pd.read_parquet(MODELS / "test_variants.parquet").assign(title=lambda t: t.dataset_id.map(titles))
kras = pd.read_parquet(ROOT / "data/kras/kras_energies.parquet").pivot_table(
    index="position", columns="measurement", values="score")       # 实验：正 = 破坏
measured = {"KRAS": ("P01116", -kras["folding"],
                     -kras[[c for c in kras.columns if c.startswith("binding")]].mean(axis=1))}
for gene, uniprot_id, abundance, activity in [
        ("CYP2C9", "P11712", "CYP2C9 abundance scores as measured by VAMP-seq",
         "CYP2C9 activity scores as measured by Click-seq"),
        ("G6PD", "P11413", "G6PD VAMP-seq Scores", "G6PD Combined Functional Scores")]:
    keys = ["uniprot_position", "alt_aa"]
    both = test[test.title == abundance].merge(test[test.title == activity][keys + ["label"]], on=keys,
                                               suffixes=("_ab", "_act"))
    both["function"] = sites.beyond_abundance(both.label_ab, both.label_act)
    by_position = both.groupby("uniprot_position")[["label_ab", "function"]].mean()
    measured[gene] = (uniprot_id, by_position.label_ab, by_position.function)
four_classes = []
for gene, (uniprot_id, fold_truth, func_truth) in measured.items():
    m = positions[positions.uniprot_id == uniprot_id].set_index("position").join(
        pd.DataFrame({"fold_truth": fold_truth, "func_truth": func_truth}), how="inner").dropna()
    out = sites.agreement(sites.position_classes(m.fold_truth, m.func_truth),
                          sites.position_classes(m.fold_cost, m.func_cost))
    four_classes.append({"蛋白": gene, "位置数": out["positions"], "一致率": out["agreement"], "随机": out["chance"],
                         "kappa": out["kappa"], "p": out["p"],
                         **{f"{name}找回率（随机）": f"{out['recall'][c][0]:.2f}（{out['recall'][c][1]:.2f}）"
                            for c, name in [(sites.FUNCTIONAL, "功能关键"), (sites.STRUCTURAL, "结构关键"),
                                            (sites.MIXED, "两者都关键"), (sites.TOLERANT, "可以改")]}})
pd.DataFrame(four_classes).round(3)

**结果怎么看**
- 三个蛋白的一致率都明显高于随机：KRAS 59% vs 42%，CYP2C9 55% vs 40%，G6PD 65% vs 38%；kappa 0.24–0.43，置换检验 p < 0.0001。
- **功能关键**的找回率是随机的 1.6–2.8 倍（KRAS 0.30 vs 0.11，CYP2C9 0.18 vs 0.11，G6PD 0.37 vs 0.13）。
- **可以改**最准，找回 79%–92%。
- 不足：前三类每一类都只找回两到四成。四类**适合用来排优先级**（先看哪些位置），**不能当成每个位置的定论**。

### 4.2 稳定性这一轴：102 个测试蛋白

**在干什么**：测试集里几乎每个蛋白都有测蛋白量的实验（大多来自 Domainome）。蛋白量只反映稳定性，所以可以直接检验：预测折叠代价高的位置，实测蛋白量是不是掉得多；"功能关键"的位置，蛋白量是不是掉得少。

**怎么算**：每个位置取实测蛋白量的平均，在每个蛋白内部排名（0 = 掉得最多）。预测的四类也在有测量的位置里重新按 25% 分。至少 20 个位置有测量的蛋白才算。

**为什么有说服力**：102 个蛋白彼此独立，可以逐个蛋白比"蛋白量和折叠代价更相关，还是和功能代价更相关"，再做符号检验。

**执行**

In [ ]:
test["reads"] = [energy.readout_kind(k, titles.get(d)) for k, d in zip(test.data_kind, test.dataset_id, strict=True)]
lost = (test[test.reads == energy.ABUNDANCE].groupby(["uniprot_id", "uniprot_position"]).label.mean()
        .rename("abundance").reset_index().rename(columns={"uniprot_position": "position"})
        .merge(positions, on=["uniprot_id", "position"]))
lost = lost[lost.groupby("uniprot_id").position.transform("size") >= 20].copy()
lost["predicted"] = lost.groupby("uniprot_id", group_keys=False)[["fold_cost", "func_cost"]].apply(
    lambda p: pd.Series(sites.position_classes(p.fold_cost, p.func_cost), index=p.index))
lost["rank"] = lost.groupby("uniprot_id").abundance.rank(pct=True)              # 0 = 蛋白量掉得最多
per_class = lost.groupby("predicted")["rank"].agg(**{
    "位置数": "size", "实测蛋白量的平均排名": "mean", "落在掉得最多的 25% 里": lambda r: (r <= 0.25).mean()}
).reindex([sites.MIXED, sites.STRUCTURAL, sites.FUNCTIONAL, sites.TOLERANT])
fold_wins = lost.groupby("uniprot_id")[["abundance", "fold_cost", "func_cost"]].apply(
    lambda p: stats.spearmanr(p.abundance, p.fold_cost)[0] > stats.spearmanr(p.abundance, p.func_cost)[0])
functional, structural = (lost[lost.predicted == c]["rank"] for c in (sites.FUNCTIONAL, sites.STRUCTURAL))
print(f"{lost.uniprot_id.nunique()} 个蛋白、{len(lost):,} 个位置")
print(f"蛋白量和折叠代价更相关的蛋白：{int(fold_wins.sum())} / {len(fold_wins)}，"
      f"符号检验 p = {stats.binomtest(int(fold_wins.sum()), len(fold_wins), alternative='greater').pvalue:.1g}")
print(f"功能关键 vs 结构关键的蛋白量排名：Mann–Whitney p = {stats.mannwhitneyu(functional, structural).pvalue:.1g}")
per_class.round(3)

**结果怎么看**
- 预测折叠代价高的两类，实测蛋白量掉得最多：两者都关键平均排名 0.23、结构关键 0.31；可以改 0.61。
- **功能关键和结构关键差得很开**（0.48 vs 0.31，p ≈ 5e-30）：功能关键的位置改了以后蛋白量掉得少，说明稳定性的影响确实被分出去了。
- **逐个蛋白**：102 个里 75 个，实测蛋白量和折叠代价更相关（符号检验 p ≈ 1e-6）。
- 不足：功能关键的位置，蛋白量还是比"可以改"掉得多（0.48 vs 0.61；落在最差 25% 的比例 29% vs 9%），拆分不完全干净。

### 4.3 功能这一轴：BioLiP 的结合残基

**在干什么**：换一个比 UniProt 密得多的标注。**BioLiP** 收集了 PDB 里所有蛋白-配体复合物的晶体结构，列出每个配体实际接触的残基，并去掉了结晶时混进去的添加剂。它是做结合位点预测时的标准数据集。

**怎么用**：
- 只取有生物学意义的配体：小分子、多肽、DNA/RNA；**金属离子不算**，因为很多金属（比如锌指里的锌）是撑结构的，不是功能
- 结构里的残基编号先和 UniProt 序列比对，再对到我们的位置（`sites.biolip_sites`），氨基酸不一致的位置丢掉
- 和第 2 步一样算富集；再逐个蛋白看，排除"只是一两个蛋白撑起来的"

**执行**（第一次运行会下载 BioLiP，约 80 MB）

In [ ]:
import urllib.request
BIOLIP = MAP / "biolip" / "BioLiP.txt.gz"
if not BIOLIP.exists():
    BIOLIP.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://zhanggroup.org/BioLiP/download/BioLiP.txt.gz", BIOLIP)
contacts = sites.biolip_sites(sites.read_biolip(BIOLIP, positions.uniprot_id.unique()), sequences.to_dict())
bound = contacts[~contacts.metal][["uniprot_id", "position"]].drop_duplicates()
covered = positions[positions.uniprot_id.isin(bound.uniprot_id)]
enriched = sites.enrichment(covered, bound)
enriched["占全部结合残基"] = enriched.annotated / enriched.annotated.sum()
marked = covered.merge(bound.assign(bound=True), on=["uniprot_id", "position"], how="left")
marked = marked.assign(bound=marked.bound.eq(True),
                       costly=marked.position_is.isin([sites.FUNCTIONAL, sites.MIXED]))
per_protein = marked.groupby("uniprot_id")[["bound", "costly"]].apply(lambda p: pd.Series({
    "inside": p[p.costly].bound.mean(), "outside": p[~p.costly].bound.mean(), "bound": p.bound.sum()}))
per_protein = per_protein[per_protein.bound >= 5]
higher = int((per_protein.inside > per_protein.outside).sum())
print(f"BioLiP：{bound.uniprot_id.nunique()} 个蛋白有结合残基，共 {len(bound):,} 个（UniProt 是 283 个）")
print(f"逐个蛋白（至少 5 个结合残基）：功能代价高的位置结合残基比例更高的 {higher} / {len(per_protein)}，"
      f"符号检验 p = {stats.binomtest(higher, len(per_protein), alternative='greater').pvalue:.1g}")
enriched.round(4)

**结果怎么看**
- BioLiP 在 45 个蛋白上给出 1,582 个结合残基，是 UniProt 注释（283 个）的 5 倍多。
- **功能关键**的位置里 12.6% 是结合残基，比值比 **4.5**；两者都关键 2.8；结构关键 0.65、可以改 0.23，都低于平均。
- 从结合残基这边看：**61%** 落在功能代价最高的那 25% 位置里（功能关键 26% + 两者都关键 35%）。
- 逐个蛋白：44 个蛋白里 **42 个**，功能代价高的位置结合残基比例更高（符号检验 p ≈ 6e-11），不是一两个蛋白撑起来的。
- 和第 2 步不同的是：UniProt 上"两者都关键"富集最强，BioLiP 上"功能关键"最强。BioLiP 多了大量表面上的配体、多肽和核酸接触，这些位置改了不影响折叠，正是"功能关键"应该抓到的。

### 4.4 更多成药口袋

**在干什么**：第 3 步只有 3 个口袋。这里事先列好 28 个"靶点 + 上市药"，在看任何结果之前就定下名单：

- **9 个变构药**：药不结合在活性位点，而是别的口袋（MEK1 cobimetinib、TYK2 deucravacitinib、IDH1 ivosidenib、IDH2 enasidenib、CFTR ivacaftor、血红蛋白 voxelotor、HIF-2α belzutifan、心肌肌球蛋白 mavacamten、钙敏感受体 cinacalcet）
- **19 个活性位点药**：药直接占住活性位点或配体口袋（激酶抑制剂、PARP、他汀、雌激素受体等）

**口袋怎么定**：在 BioLiP 里找这个药和这个靶点的共晶结构（按 PDB 化学字典和 DrugBank 的药名匹配，`sites.ligand_names`），取分辨率最好的那个结构，药接触的残基就是口袋（`sites.drug_pocket`）。找不到共晶结构的药如实列出来，不换药。

**怎么判断**：和第 3 步一样，口袋残基在自己蛋白里按功能代价排第几（百分位，0 = 最敏感）。同时给出**折叠代价**和 **ESM-2 单分数**的百分位作对照。

这些靶点大多不在第 1 步的地图里，所以先用同一个模型给它们打分，另存一份（`pocket_targets.parquet`），不改湖里的地图。

**执行**

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "databricks" / "src"))
from lakehouse import language_model as lm
DRUGS = {   # 靶点：(UniProt, 药, 口袋类型)，事先定好
    "MAP2K1": ("Q02750", "cobimetinib", "变构"), "TYK2": ("P29597", "deucravacitinib", "变构"),
    "IDH1": ("O75874", "ivosidenib", "变构"), "IDH2": ("P48735", "enasidenib", "变构"),
    "CFTR": ("P13569", "ivacaftor", "变构"), "HBA1": ("P69905", "voxelotor", "变构"),
    "EPAS1": ("Q99814", "belzutifan", "变构"), "MYH7": ("P12883", "mavacamten", "变构"),
    "CASR": ("P41180", "cinacalcet", "变构"),
    "BTK": ("Q06187", "ibrutinib", "活性位点"), "EGFR": ("P00533", "erlotinib", "活性位点"),
    "BRAF": ("P15056", "vemurafenib", "活性位点"), "CDK6": ("Q00534", "palbociclib", "活性位点"),
    "PARP1": ("P09874", "olaparib", "活性位点"), "PDE5A": ("O76074", "sildenafil", "活性位点"),
    "HMGCR": ("P04035", "atorvastatin", "活性位点"), "PTGS2": ("P35354", "celecoxib", "活性位点"),
    "ESR1": ("P03372", "raloxifene", "活性位点"), "F10": ("P00742", "rivaroxaban", "活性位点"),
    "DPP4": ("P27487", "sitagliptin", "活性位点"), "ACHE": ("P22303", "donepezil", "活性位点"),
    "CA2": ("P00918", "acetazolamide", "活性位点"), "DHFR": ("P00374", "methotrexate", "活性位点"),
    "BCL2": ("P10415", "venetoclax", "活性位点"), "TTR": ("P02766", "tafamidis", "活性位点"),
    "PPARG": ("P37231", "rosiglitazone", "活性位点"), "DHODH": ("Q02127", "teriflunomide", "活性位点"),
    "MAOB": ("P27338", "safinamide", "活性位点"),
}
targets = sites.biolip_sites(sites.read_biolip(BIOLIP, [u for u, _, _ in DRUGS.values()]), sequences.to_dict())
targets = targets[~targets.metal]
names = sites.ligand_names(targets.ligand.unique())
drug_pockets, missing = {}, []
for gene, (uniprot_id, drug, kind) in DRUGS.items():
    mine = names[names.ligand.isin(targets[targets.uniprot_id == uniprot_id].ligand)]
    hit = mine[mine.drugbank.eq(drug) | mine.name.str.contains(drug) | mine.synonyms.str.contains(drug)]
    if hit.empty:
        missing.append(f"{gene}（{drug}）")
        continue
    drug_pockets[gene] = (uniprot_id, drug, kind, *sites.drug_pocket(targets, uniprot_id, set(hit.ligand)))
print(f"找到共晶结构 {len(drug_pockets)} 个；BioLiP 里没有的：{'、'.join(missing)}")
lm_model, tokenizer = lm.load()
residues = pd.read_parquet(MODELS / "proteome_residues.parquet",
                           filters=[("uniprot_id", "in", [p[0] for p in drug_pockets.values()])])
every = sites.score(models, residues, sequences,
                    lambda seqs: lm.read_proteins(seqs, lm_model, tokenizer, embeddings=True))
target_map = sites.classify(every).merge(
    every.groupby(["uniprot_id", "position"]).esm_delta.mean().rename("esm").reset_index())
target_map.to_parquet(MAP / "pocket_targets.parquet")
trained_on = set(split_table.loc[split_table.split == "train", "uniprot_id"])
ranked = []
for gene, (uniprot_id, drug, kind, pdb, pocket) in drug_pockets.items():
    r = {c: sites.pocket_rank(target_map, uniprot_id, pocket, column=c) for c in ("func_cost", "fold_cost", "esm")}
    ranked.append({"靶点": gene, "药": drug, "口袋": kind, "结构": pdb, "残基数": r["func_cost"]["pocket residues"],
                   "功能代价百分位": r["func_cost"]["median percentile"], "p": r["func_cost"]["p"],
                   "折叠代价百分位": r["fold_cost"]["median percentile"], "ESM-2 百分位": r["esm"]["median percentile"],
                   "07a 训练侧": uniprot_id in trained_on})
ranked = pd.DataFrame(ranked)
for kind, g in ranked.groupby("口袋", sort=False):
    print(f"{kind}：{len(g)} 个口袋；功能代价排进前 25% 的 {int((g['功能代价百分位'] <= 0.25).sum())} 个，"
          f"p < 0.05 的 {int((g.p < 0.05).sum())} 个；中位百分位 功能代价 {g['功能代价百分位'].median():.2f}、"
          f"折叠代价 {g['折叠代价百分位'].median():.2f}、ESM-2 {g['ESM-2 百分位'].median():.2f}")
better = int((ranked["功能代价百分位"] < ranked["ESM-2 百分位"]).sum())
print(f"功能代价比 ESM-2 单分数把口袋排得更靠前的：{better} / {len(ranked)}，"
      f"符号检验 p = {stats.binomtest(better, len(ranked), alternative='greater').pvalue:.1g}")
ranked.round(3)

**结果怎么看**
- 28 个药里 21 个在 BioLiP 里有共晶结构。7 个没有（TYK2、血红蛋白、HIF-2α、心肌肌球蛋白、COX-2、乙酰胆碱酯酶、BCL2），如实列出，没有换药。
- **活性位点药（16 个）**：15 个显著（p < 0.05），12 个排进自己蛋白功能代价最高的前 25%；中位百分位 0.14。
- **变构药（5 个）**：MEK1（cobimetinib，0.04）和钙敏感受体（cinacalcet，0.24）显著；IDH1、IDH2、CFTR 不显著。加上第 3 步的 3 个，**变构口袋一共 8 个，中了 4 个**。
- **和 ESM-2 单分数比**：21 个口袋里 **19 个**，功能代价把口袋排得比 ESM-2 更靠前（符号检验 p ≈ 0.0001）。差距在变构口袋上最大（中位百分位 0.28 vs 0.47）。
- **和折叠代价比**：口袋的折叠代价百分位中位数是 0.32–0.38，远不如功能代价靠前。点亮口袋的是功能这一轴，不只是"埋得深"。
- **边界**：没中的 IDH1、IDH2 口袋在两个亚基的交界处，CFTR 的口袋在膜里，和第 3 步的 SHP2 一样，是靠改变构象起作用的位点。"折好之后还能不能干活"这个量对它们不敏感。
- BTK、BRAF、HMGCR、PPARG 在 07a 的训练侧，但 14c 的能量模型没用它们的数据训练。

### 面试怎么说
> "I widened the evidence beyond UniProt's 283 annotated sites. On 102 held-out proteins, measured abundance tracks the folding cost in 75 (p 1e-6). BioLiP's 1,582 ligand-contact residues are enriched 4.5-fold among functionally costly positions, in 42 of 44 proteins. For 21 approved drugs with a co-crystal structure, the functional cost ranks the pocket ahead of ESM-2 in 19; allosteric pockets at subunit interfaces remain a blind spot."

---

## 第 5 步 · 地图进数据湖

### 在干什么
把位置地图写进 S3 的 lake 桶，在 Glue 里登记成一张表，这样它和 10a 的预测表一样可以用 SQL 查。

### 背景
一张放在笔记本电脑上的 parquet 文件不是产品。放进湖里之后：分析师可以直接写 SQL（"TP53 上有哪些功能关键但结构可改的位置"）、API 可以查、agent 可以调用。这是把一章的科学结果**交付**出去。

表的定义写在 `infra/dev/lake.tf`（`sites`，和 `variant_effect`、`proteins` 并列），由 Terraform 创建。

### 执行
先建表（Terraform），再上传数据。

In [ ]:
run("terraform -chdir=infra/dev plan -no-color -out=sites.tfplan | tail -5")

In [ ]:
import boto3, json
run("terraform -chdir=infra/dev apply -no-color sites.tfplan | tail -3")
aws = boto3.Session(profile_name="qafi", region_name="eu-central-1")
s3, client = aws.client("s3"), aws.client("athena")
buckets = json.loads(subprocess.run("terraform -chdir=infra/dev output -json bucket_names",
                                    shell=True, capture_output=True, text=True, check=True).stdout)
release = pd.Timestamp.today().strftime("%Y-%m-%d")
out = MAP / "sites.parquet"
positions.astype({"position": "int32", "fold_cost": "float32", "func_cost": "float32"}).to_parquet(out, index=False)
s3.upload_file(str(out), buckets["lake"], f"sites/release={release}/sites.parquet")
lake.query(client, f"ALTER TABLE sites ADD IF NOT EXISTS PARTITION (release = '{release}')",
           "qafi-dev", "qafi_dev_lake")
print(f"已上传 {len(positions):,} 行 → s3://{buckets['lake']}/sites/release={release}/")

In [ ]:
summary_sql = ("SELECT position_is, count(*) AS positions, round(avg(fold_cost), 3) AS mean_fold_cost, "
               "round(avg(func_cost), 3) AS mean_func_cost "
               "FROM sites WHERE release = ? GROUP BY position_is ORDER BY positions DESC")
# Athena reads an execution parameter as SQL, so a string has to arrive quoted: athena.literal does that
lake.query(client, summary_sql, "qafi-dev", "qafi_dev_lake", parameters=[athena.literal(release)])

### 结果怎么看
同一张地图，现在用 SQL 就能查。一个分析师想问"KRAS 上哪些位置功能关键但结构上可以改"，不用找我们要文件，直接：

```sql
SELECT position, func_cost FROM sites
WHERE release = '…' AND uniprot_id = 'P01116' AND position_is = 'functional'
ORDER BY func_cost
```

在 KRAS 上这条查询返回的前几位是 **75、17、12、38、117、35、123、13**。第 **12** 和第 **13** 位正是 KRAS 最著名的两个致癌突变热点，而第 12 位也在 sotorasib 的 switch-II 口袋残基名单里——模型没被告诉过任何一件事。

---

## 第 6 步 · 接进 API 和 agent

### 在干什么
让 10b 的对外 API 和 12 号的 agent 也能用这张地图。

### 背景
放进湖里，分析师能用 SQL 查。但 API 和 agent 面向的是不写 SQL 的人——以及其他程序。新增一个地址：

```
GET /v1/proteins/{protein}/sites?position_is=functional
```

和 10a 的做法一样，API 读的是**视图** `sites_current`，不是某个具体版本；换一版地图只要移动视图，不用动代码。agent 那边新增一个工具 `get_site_map`，并且给护栏的依据检查写了对应的文字（12 号的教训：JSON 判不准，要写成句子）。

实现：`src/qafi_platform/api.py` 加一个路由，`src/qafi_platform/agent.py` 加一个工具，各自配了测试（API 21 个、agent 14 个）。镜像重建后由 Terraform 按摘要部署。

### 执行

In [ ]:
import boto3, json
from qafi_platform import api_client
API = json.loads(subprocess.run("terraform -chdir=infra/dev output -json api",
                                shell=True, capture_output=True, text=True, check=True).stdout)
session = boto3.Session(profile_name="qafi", region_name="eu-central-1")
status, body, seconds = api_client.get(session, f"{API['url'].rstrip('/')}/v1/proteins/KRAS/sites")
print(f"{status} in {seconds:.1f}s | KRAS 共 {body['mapped']} 个位置")
pd.DataFrame(body["sites"]).head(8)

### 结果怎么看
线上 API 返回 KRAS 全部 189 个位置，按功能代价从高到低排好。

一个值得看的细节：KRAS 最著名的三个致癌突变热点 **G12、G13、Q61**，在这个排序里分别排第 **11、26、44** 位（前 6%、14%、23%）——第 12 位归为 mixed（两个能量都高），13 和 61 是 functional。模型只见过突变的测量值，从没被告知哪些位置和癌症有关。

agent 也能用了。问它"KRAS 哪些位置功能敏感但结构上能改"，它会调用 `get_site_map`，用返回的数字回答，并标注出处。

---

## 第 7 步 · 记账

In [ ]:
enrichment = sites.enrichment(positions, annotated).set_index("position_is")
pockets = pd.DataFrame(found)
metrics = {f"odds_ratio_{k}": float(enrichment.loc[k, "odds ratio"]) for k in enrichment.index
           if np.isfinite(enrichment.loc[k, "odds ratio"])}
metrics |= {f"pocket_percentile_{i}": float(r["口袋的中位百分位"]) for i, r in pockets.iterrows()}
metrics |= {"positions": len(positions), "proteins": int(positions.uniprot_id.nunique())}
tracking.experiment_run(
    EXPERIMENT, "site_map",
    params={"proteins": int(positions.uniprot_id.nunique()), "quantile": 0.25,
            "readout_proteins": len(paired), "pockets": len(POCKETS), "release": release},
    metrics=metrics,
    tables={"enrichment": enrichment.reset_index(), "pockets": pockets},
    chapter="14d")
print("已记录")

第 4 步的四项检验另记一笔：

In [ ]:
tracking.experiment_run(
    EXPERIMENT, "site_map_checks",
    params={"truth_4_1": "KRAS energies; CYP2C9, G6PD abundance + activity beyond abundance",
            "truth_4_3": "BioLiP non-metal ligand contacts", "pockets_listed": len(DRUGS), "pockets_found": len(drug_pockets)},
    metrics={**{f"kappa_{r['蛋白']}": float(r["kappa"]) for r in four_classes},
             "abundance_fold_wins": int(fold_wins.sum()), "abundance_proteins": len(fold_wins),
             **{f"biolip_odds_{k}": float(v) for k, v in enriched.set_index("position_is")["odds ratio"].items()},
             "pockets_top_quarter": int((ranked["功能代价百分位"] <= 0.25).sum()),
             "pockets_significant": int((ranked.p < 0.05).sum()), "pockets_func_beats_esm": better},
    tables={"four_classes": pd.DataFrame(four_classes), "abundance_by_class": per_class.reset_index(),
            "biolip": enriched, "drug_pockets": ranked},
    chapter="14d")
print("已记录")

---

## 这一章做完了什么

- 把 14c 的两个能量按位置平均，给 118 个蛋白的 103,744 个位置贴上**结构关键 / 功能关键 / 两者 / 可以改**
- **对照 UniProt 注释**：功能关键富集 2.7 倍、两者都关键 9.3 倍，而结构关键被稀释（0.21 倍）——一个分数做不出这种分化
- **三个已成药的变构口袋**：KRAS switch-II 和 ABL1 肉豆蔻酸口袋都排进自己蛋白的前 23%（p ≈ 0.001）；SHP2 的构象开关型通道没信号，这是方法的边界
- **更多检验**（全部是模型没见过的蛋白）：
  - 和实测四类对照（KRAS、CYP2C9、G6PD）：一致率都高于随机，kappa 0.24–0.43；适合排优先级，不能当定论
  - 稳定性轴：102 个蛋白里 75 个，实测蛋白量和折叠代价更相关（p ≈ 1e-6）
  - 功能轴：BioLiP 的 1,582 个结合残基在功能关键位置富集 4.5 倍，44 个蛋白里 42 个方向一致
  - 21 个上市药的口袋：活性位点 16 个中 15 个、变构 5 个中 2 个显著；19 个口袋功能代价比 ESM-2 排得更前。变构口袋在亚基交界和膜里的，是方法的边界
- 地图进了 S3 湖和 Glue（能用 Athena 查）、对外 API（`/v1/proteins/{protein}/sites`）和 agent（`get_site_map` 工具）

## 下一步

**14e · 药理伴侣**：这一章说了"折叠坏了可以试试扶住它的小分子"。14e 真的去验证一次——GLA 蛋白的突变里，哪些能被已上市的药 migalastat 救回来。

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14d site map: what each position is for' && git log --oneline -1")